# EDA — Datathon 2026 Round 1

**Bài toán:** dự báo `Revenue` và `COGS` theo ngày cho **2023-01-01 → 2024-07-01** (548 ngày),
dựa trên lịch sử **2012-07-04 → 2022-12-31** (3.833 ngày, không thiếu ngày nào).

Notebook này đi tìm **cấu trúc sinh dữ liệu**, không xây model. Năm phát hiện chính:

1. `sales.csv` **tái tạo được chính xác** từ `order_items × orders × products` → target là hàm xác định của dữ liệu giao dịch.
2. Có **đứt gãy cấu trúc cuối 2018**: doanh thu rơi ~40% rồi đi ngang. Tăng trưởng gộp 2013–2022 mà baseline dùng (**−3,8%/năm**) là sai.
3. **Mùa vụ theo tháng rất mạnh** (đỉnh T4–T6, đáy T12–T1) và **hiệu ứng ngày-trong-tháng** cực lớn (dốc tăng dần tới cuối tháng).
4. **Thứ trong tuần gần như vô nghĩa** (R² = 0,008) — baseline bỏ qua nó là đúng.
5. **Tháng 8 có chu kỳ 2 năm cực mạnh**: năm lẻ có siêu khuyến mãi "Urban Blowout" làm doanh thu −37% và **COGS > Revenue** (tỷ lệ ~1,36). **2023 là năm lẻ và tháng 8/2023 nằm trong vùng dự báo.**

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)
DATA = 'data'          # các file .csv nằm cùng thư mục với notebook này
R = lambda f, **k: pd.read_csv(f'{DATA}/{f}', low_memory=False, **k)

sales  = R('sales.csv',  parse_dates=['Date'])
sub    = R('sample_submission.csv', parse_dates=['Date'])
orders = R('orders.csv', parse_dates=['order_date'])
items  = R('order_items.csv')
prod   = R('products.csv')

for c, src in [('year','Date.dt.year'), ('month','Date.dt.month'), ('day','Date.dt.day'),
               ('dow','Date.dt.dayofweek')]:
    sales[c] = eval('sales.'+src)
sales['ratio'] = sales.COGS / sales.Revenue

print('sales   :', sales.Date.min().date(), '->', sales.Date.max().date(), f'({len(sales):,} ngày)')
print('ngày thiếu:', len(set(pd.date_range(sales.Date.min(), sales.Date.max())) - set(sales.Date)))
print('submission:', sub.Date.min().date(), '->', sub.Date.max().date(), f'({len(sub):,} ngày)')

---
## 1 — `sales.csv` tái tạo được **chính xác** từ dữ liệu giao dịch

Đây là kiểm tra quan trọng nhất: nếu target suy ra được từ các bảng con thì mọi bảng đều là
nguồn feature hợp lệ; nếu không thì đây chỉ là bài time-series đơn biến.

Công thức đúng (đã dò qua mọi tổ hợp filter):

```
Revenue(d) = Σ  quantity × unit_price          ← TẤT CẢ đơn, KHÔNG trừ discount_amount
COGS(d)    = Σ  quantity × products.cogs       ← TẤT CẢ đơn
```

Lưu ý hai điểm phản trực giác: **không lọc `order_status`** (kể cả `cancelled`/`returned`) và
**không trừ `discount_amount`**.

In [ ]:
m = (items.merge(orders[['order_id','order_date','order_status']], on='order_id')
          .merge(prod[['product_id','cogs']], on='product_id'))
m['gross']    = m.quantity * m.unit_price
m['net']      = m.gross - m.discount_amount
m['cogs_amt'] = m.quantity * m.cogs

def reconcile(mask, label):
    g = m[mask].groupby('order_date').agg(rev_g=('gross','sum'), rev_n=('net','sum'), cg=('cogs_amt','sum'))
    j = sales.set_index('Date').join(g, how='inner')
    return pd.Series({
        'Rev  (gross)': (j.rev_g/j.Revenue).median(),
        'Rev  (net)'  : (j.rev_n/j.Revenue).median(),
        'COGS'        : (j.cg  /j.COGS   ).median(),
    }, name=label)

tbl = pd.concat([
    reconcile(m.order_status.notna(),                            'TẤT CẢ đơn'),
    reconcile(~m.order_status.isin(['cancelled']),               'bỏ cancelled'),
    reconcile(~m.order_status.isin(['cancelled','returned']),    'bỏ cancelled+returned'),
    reconcile(m.order_status.eq('delivered'),                    'chỉ delivered'),
], axis=1).T
print('Tỷ lệ  tái_tạo / thực_tế  (1.0000 = khớp hoàn hảo)\n')
print(tbl.round(4).to_string())

In [ ]:
# Độ chính xác của công thức thắng cuộc
g = m.groupby('order_date').agg(rev=('gross','sum'), cg=('cogs_amt','sum'))
j = sales.set_index('Date').join(g)
print(f'Sai số tương đối lớn nhất — Revenue : {((j.Revenue-j.rev).abs()/j.Revenue).max():.3e}')
print(f'Sai số tương đối lớn nhất — COGS    : {((j.COGS  -j.cg ).abs()/j.COGS   ).max():.3e}')
print('\n→ Revenue khớp tới sai số dấu phẩy động (2e-16). COGS khớp tới 1e-8 (do sales.csv làm tròn 2 chữ số).')
print('→ TARGET LÀ HÀM XÁC ĐỊNH CỦA order_items + orders + products.')

### Ý nghĩa

`Revenue = Σ qty × unit_price` nên nó tách được thành các nhân tố có thể mô hình hoá riêng:

$$\text{Revenue}(d) = \underbrace{N_{\text{đơn}}(d)}_{\text{lưu lượng}} \times \underbrace{\frac{\text{units}}{\text{đơn}}}_{\text{giỏ hàng}} \times \underbrace{\overline{\text{unit\_price}}(d)}_{\text{giá}}$$

Và `COGS` chỉ phụ thuộc **units × giá vốn** — **không** phụ thuộc `unit_price`.
Đây chính là lý do tháng 8 năm lẻ có `COGS > Revenue` (phần 5).

In [ ]:
g2 = m.groupby('order_date').agg(rev=('gross','sum'), cogs=('cogs_amt','sum'),
                                 units=('quantity','sum'), n_orders=('order_id','nunique'))
g2['giá_TB/unit']   = g2.rev  / g2.units
g2['vốn_TB/unit']   = g2.cogs / g2.units
g2['units/đơn']     = g2.units/ g2.n_orders
g2['DT/đơn']        = g2.rev  / g2.n_orders
yr = g2.groupby(g2.index.year).agg({'rev':'sum','cogs':'sum','n_orders':'sum','units':'sum',
                                    'giá_TB/unit':'mean','vốn_TB/unit':'mean',
                                    'units/đơn':'mean','DT/đơn':'mean'})
yr[['rev','cogs']] /= 1e6
print('Phân rã theo năm (rev/cogs: triệu)\n')
print(yr.round(1).to_string())

---
## 2 — Xu hướng: có **đứt gãy cấu trúc**, không phải tăng trưởng đều

Baseline tính tăng trưởng gộp trung bình nhân 2013–2022 rồi ngoại suy. Nhưng chuỗi **không**
có một xu hướng duy nhất — nó vỡ vào cuối 2018.

In [ ]:
ann = sales.groupby('year')[['Revenue','COGS']].sum()/1e6
g   = ann.loc[2013:2022].pct_change().dropna()
geo = {c: (1+g[c]).prod()**(1/len(g)) for c in ['Revenue','COGS']}

print('Tổng theo năm (triệu):'); print(ann.round(0).to_string())
print(f"\nTăng trưởng gộp 2013–2022 (baseline dùng) : Revenue {(geo['Revenue']-1)*100:+.2f}%/năm")
print(f"Nhưng CAGR 2019→2022                      : Revenue {((ann.Revenue[2022]/ann.Revenue[2019])**(1/3)-1)*100:+.2f}%/năm")
print('\n→ Baseline sẽ suy giảm mức dự báo ~3,8%/năm trong khi thực tế 2019–2022 đi ngang / nhích lên.')

fig, ax = plt.subplots(1, 2, figsize=(15,4))
ax[0].bar(ann.index, ann.Revenue, color='steelblue'); ax[0].set_title('Doanh thu theo năm (triệu)')
ax[0].axvspan(2018.5, 2022.5, color='orange', alpha=.15); ax[0].text(2020.5, ann.Revenue.max()*.9, 'chế độ mới', ha='center')
mm = sales.set_index('Date').Revenue.resample('MS').sum()/1e6
ax[1].plot(mm.index, mm.values, lw=1); ax[1].axvline(pd.Timestamp('2018-09-01'), color='r', ls='--')
ax[1].set_title('Doanh thu theo tháng — đứt gãy ~T9/2018')
plt.tight_layout(); plt.show()

In [ ]:
mm2 = sales.set_index('Date').Revenue.resample('MS').sum()/1e6
print('Quanh điểm gãy (triệu):')
print(mm2['2018-06':'2019-06'].round(0).to_string())
print('\nSố đơn/năm:'); print(g2.groupby(g2.index.year).n_orders.sum().to_string())
print('\n→ Số đơn rơi 69.5k (2018) → 41.6k (2019) rồi ổn định ~35k. Giá trị/đơn thì TĂNG đều.')
print('→ Chỉ nên fit mức (level) trên dữ liệu 2019+; mùa vụ thì có thể dùng nhiều năm hơn.')

---
## 3 — Mùa vụ trong năm: rất mạnh và ổn định

Đỉnh **T4–T6** (~1,5× trung bình), đáy **T12–T1** (~0,6×). Biên độ 2,6 lần.
Chỉ số mùa vụ khá ổn định qua các năm — **trừ tháng 8** (xem phần 5).

In [ ]:
sales['rev_n'] = sales.Revenue / sales.groupby('year').Revenue.transform('mean')
mi = sales.pivot_table(index='month', columns='year', values='rev_n', aggfunc='mean')
prof = pd.DataFrame({'tất cả năm': mi.mean(axis=1), '3 năm gần nhất': mi[[2020,2021,2022]].mean(axis=1),
                     'CV giữa các năm': mi.std(axis=1)/mi.mean(axis=1)})
print('Chỉ số mùa vụ theo tháng (Revenue / trung bình năm)\n'); print(prof.round(3).to_string())
print('\n→ CV cao bất thường ở T8 (0,24) và T12 (0,23); các tháng khác < 0,15.')

fig, ax = plt.subplots(1, 2, figsize=(15,4))
ax[0].plot(prof.index, prof['tất cả năm'], 'o-', label='tất cả năm')
ax[0].plot(prof.index, prof['3 năm gần nhất'], 's--', label='2020–2022')
ax[0].axhline(1, color='grey', lw=.5); ax[0].set_xlabel('tháng'); ax[0].legend(); ax[0].set_title('Chỉ số mùa vụ tháng')
im = ax[1].imshow(mi.T, aspect='auto', cmap='RdYlBu_r', extent=[.5,12.5,2022.5,2011.5])
ax[1].set_title('Heatmap chỉ số mùa vụ (hàng = năm)'); ax[1].set_xlabel('tháng'); plt.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.show()

---
## 4 — Hiệu ứng **ngày-trong-tháng**: tín hiệu bị baseline bỏ sót nhiều nhất

Trong mọi tháng, doanh thu **tăng dần đều** từ ~0,6× (ngày 4–5) lên ~1,8× (ngày 28–31),
kèm một nhịp cao ở ngày 1–3 (đuôi của đợt đẩy cuối tháng trước).
Đây là hiệu ứng **lớn hơn nhiều** so với thứ-trong-tuần.

In [ ]:
sales['rev_m'] = sales.Revenue / sales.groupby(['year','month']).Revenue.transform('mean')
dom = sales.groupby('day').rev_m.mean()

# so sánh sức giải thích
y = np.log(sales.Revenue.values)
def r2(cols):
    p = sales.assign(_y=y).groupby(cols)._y.transform('mean')
    return 1 - ((y-p)**2).sum()/((y-y.mean())**2).sum()

print('R² của log(Revenue) khi chỉ dùng nhóm lịch:\n')
for cols in [['dow'], ['year'], ['month'], ['day'], ['month','dow'], ['month','day'],
             ['year','month'], ['year','month','dow'], ['year','month','day']]:
    print(f'  {str(cols):26s} R² = {r2(cols):.4f}')
print('\n→ dow: 0,008 (vô nghĩa).  day-of-month: cao hơn cả tháng.  month+day: 0,623.')

fig, ax = plt.subplots(1, 2, figsize=(15,4))
ax[0].bar(dom.index, dom.values, color='teal'); ax[0].axhline(1, color='r', lw=.8)
ax[0].set_title('Chỉ số ngày-trong-tháng'); ax[0].set_xlabel('ngày')
dw = sales.groupby('dow').rev_n.mean()
ax[1].bar(['T2','T3','T4','T5','T6','T7','CN'], dw.values, color='indianred'); ax[1].axhline(1, color='k', lw=.8)
ax[1].set_ylim(.8,1.15); ax[1].set_title('Chỉ số thứ-trong-tuần (biên độ rất hẹp)')
plt.tight_layout(); plt.show()

In [ ]:
print('sample_submission có tái hiện hiệu ứng ngày-trong-tháng không?\n')
sub['rn'] = sub.Revenue / sub.groupby(sub.Date.dt.to_period('M')).Revenue.transform('mean')
cmp = pd.DataFrame({'lịch sử': dom,
                    'sample_submission': sub[sub.Date < '2024-07-01'].groupby(sub.Date.dt.day).rn.mean()})
print(cmp.round(2).to_string())
print('\n→ Có, gần như trùng khớp. sample_submission KHÔNG phải số ngẫu nhiên —')
print('  nó là một dự báo baseline đã nắm được mùa vụ tháng + ngày-trong-tháng.')

---
## 5 — 🎯 Phát hiện lớn nhất: **tháng 8 có chu kỳ 2 năm**

Tháng 8 **năm chẵn** và **năm lẻ** là hai chế độ hoàn toàn khác nhau, lặp lại không sai một lần
trong suốt 11 năm:

| | Doanh thu T8 (chỉ số) | COGS/Revenue |
|---|---|---|
| Năm **chẵn** (2012,14,16,18,20,22) | **1,25** | **0,80** |
| Năm **lẻ** (2013,15,17,19,21) | **0,78** (−37%) | **1,36** ← *lỗ* |

**2023 là năm lẻ, và tháng 8/2023 nằm trong vùng dự báo.**

In [ ]:
aug = sales[sales.month==8].groupby('year').agg(rev=('Revenue','sum'), cogs=('COGS','sum'))
aug['tỷ lệ COGS/Rev'] = aug.cogs/aug.rev
aug['chỉ số DT']      = sales[sales.month==8].groupby('year').rev_n.mean()
aug['loại năm']       = np.where(aug.index % 2 == 0, 'CHẴN', 'LẺ')
print((aug.assign(rev=aug.rev/1e6, cogs=aug.cogs/1e6)).round(3).to_string())

print('\nSo sánh chẵn/lẻ trên MỌI tháng (để chứng minh chỉ T8 bị ảnh hưởng):')
par = sales.pivot_table(index='month', columns=sales.year % 2, values='rev_n', aggfunc='mean')
par.columns = ['năm chẵn','năm lẻ']; par['lệch %'] = (par['năm lẻ']/par['năm chẵn']-1)*100
rr = sales.pivot_table(index='month', columns=sales.year % 2, values='ratio', aggfunc='mean')
par['ratio chẵn'], par['ratio lẻ'] = rr[0], rr[1]
print(par.round(3).to_string())
print('\n→ Mọi tháng lệch < 8%. Tháng 8 lệch −37% và ratio nhảy 0,80 → 1,36.')

### Cơ chế: khuyến mãi **"Urban Blowout"** chạy 2 năm một lần

`promotions.csv` cho thấy promo `Urban Blowout` (giảm giá `fixed`, hàng Streetwear)
chạy **30/07 → 02/09 chỉ trong các năm lẻ**: 2013, 2015, 2017, 2019, 2021.

Điểm mấu chốt: đợt giảm giá này **hạ thẳng `unit_price`** (không đi qua `discount_amount`).
Mà `Revenue = qty × unit_price` còn `COGS = qty × products.cogs` — **giá vốn không đổi**.
Nên doanh thu sụp còn giá vốn giữ nguyên → **COGS vượt Revenue**.

In [ ]:
pm = R('promotions.csv', parse_dates=['start_date','end_date'])
ub = pm[pm.promo_name.str.contains('Urban Blowout')]
print('Các đợt "Urban Blowout" trong lịch sử:')
print(ub[['promo_id','promo_name','promo_type','discount_value','start_date','end_date',
          'applicable_category','min_order_value']].to_string(index=False))
print('\n→ Chỉ có ở năm LẺ. Chu kỳ 2 năm → dự kiến "Urban Blowout 2023": 2023-07-30 → 2023-09-02.')
print('  (promotions.csv chỉ tới 2022 nên phải ngoại suy quy luật này.)')

mm3 = m.assign(y=m.order_date.dt.year, mo=m.order_date.dt.month)
a = mm3[mm3.mo==8].groupby('y').agg(units=('quantity','sum'), rev=('gross','sum'), cg=('cogs_amt','sum'),
                                    tỷ_lệ_dòng_có_promo=('promo_id', lambda x: x.notna().mean()))
a['giá_TB/unit'] = a.rev/a.units; a['vốn_TB/unit'] = a.cg/a.units
a['loại năm'] = np.where(a.index % 2 == 0, 'CHẴN', 'LẺ')
print('\nCơ chế — tháng 8 theo năm:')
print(a[['units','giá_TB/unit','vốn_TB/unit','tỷ_lệ_dòng_có_promo','loại năm']].round(3).to_string())
print('\n→ Số units gần như KHÔNG đổi. Chỉ có giá bán/unit sụp ~40% trong năm lẻ.')
print('→ Giá vốn/unit đi lên đều đặn, hoàn toàn không phản ứng với promo.')

In [ ]:
print('sample_submission xử lý tháng 8/2023 thế nào?\n')
sm = sub.groupby(sub.Date.dt.to_period('M')).agg(rev=('Revenue','sum'), cogs=('COGS','sum'))
sm['ratio'] = sm.cogs/sm.rev; sm['rev_triệu'] = sm.rev/1e6
print(sm[['rev_triệu','ratio']].round(3).to_string())

odd_ratio = aug[aug['loại năm']=='LẺ']['tỷ lệ COGS/Rev'].mean()
odd_idx   = aug[aug['loại năm']=='LẺ']['chỉ số DT'].mean()
mmr       = sm.loc['2023-01':'2023-12','rev'].mean()
sub_idx   = sm.loc['2023-08','rev']/mmr
print(f"\n  T8 năm lẻ trong lịch sử : chỉ số DT = {odd_idx:.3f} , COGS/Rev = {odd_ratio:.3f}")
print(f"  sample_submission 08/2023: chỉ số DT = {sub_idx:.3f} , COGS/Rev = {sm.loc['2023-08','ratio']:.3f}")

# quy ra mức tuyệt đối để thấy sai số thực sự nằm ở đâu
exp_rev, = [odd_idx*mmr]; exp_cogs = exp_rev*odd_ratio
print(f"\n  Revenue T8/2023: sample = {sm.loc['2023-08','rev']/1e6:5.1f}tr | quy luật năm lẻ hàm ý"
      f" {exp_rev/1e6:5.1f}tr → lệch {(sm.loc['2023-08','rev']/exp_rev-1)*100:+.1f}%")
print(f"  COGS    T8/2023: sample = {sm.loc['2023-08','cogs']/1e6:5.1f}tr | quy luật năm lẻ hàm ý"
      f" {exp_cogs/1e6:5.1f}tr → lệch {(sm.loc['2023-08','cogs']/exp_cogs-1)*100:+.1f}%")

print('\n→ sample_submission DỰ BÁO THỪA doanh thu T8/2023 ~34%. Đây là lợi thế lớn nhất.')
print('→ Với COGS thì hai sai số TRIỆT TIÊU nhau (thừa doanh thu × thiếu tỷ lệ) nên mức COGS')
print('  tình cờ gần đúng. Đừng nhầm là "sample đã xử lý đúng tháng 8" — nó đúng do may mắn,')
print('  và chỉ đúng ở tổng tháng, không đúng theo từng ngày.')
print('→ Nguyên nhân: lấy trung bình cả năm chẵn lẫn lẻ nên rơi vào giữa hai chế độ.')
print('  Baseline notebook (trung bình profile mọi năm) mắc y hệt lỗi này.')

---
## 6 — `COGS/Revenue`: nên mô hình hoá như một **tỷ lệ**, không phải mức độc lập

Tỷ lệ này rất có cấu trúc và dễ đoán hơn nhiều so với việc dự báo hai chuỗi tương quan cao riêng rẽ.

In [ ]:
print('Tỷ lệ COGS/Revenue theo tháng × năm:')
print(sales.pivot_table(index='year', columns='month', values='ratio', aggfunc='mean').round(3).to_string())
print('\nSố ngày LỖ (COGS > Revenue) theo tháng:')
print((sales.groupby('month').apply(lambda x: (x.ratio>1).mean())*100).round(1).to_string())
print('\n→ Ba vùng: T8 năm lẻ (~1,36), T12 (~0,98–1,02 mọi năm), còn lại ~0,80–0,88.')
print('→ Biên lợi nhuận đang bị bào mòn: ratio TB 2020 = %.3f → 2022 = %.3f'
      % (sales[sales.year==2020].ratio.mean(), sales[sales.year==2022].ratio.mean()))

fig, ax = plt.subplots(figsize=(14,3.5))
ax.plot(sales.Date, sales.ratio, lw=.5); ax.axhline(1, color='r', ls='--')
ax.set_title('COGS / Revenue theo ngày — các đỉnh nhọn là T8 năm lẻ'); plt.tight_layout(); plt.show()

---
## 7 — Các bảng phụ: dùng được đến đâu?

**Ràng buộc quyết định: mọi bảng phụ đều kết thúc 2022-12-31.** Không có bảng nào phủ
2023–2024, nên **không dùng làm biến ngoại sinh cho dự báo được** — chúng chỉ giúp *giải thích*
cấu trúc (như đã dùng ở phần 5), không giúp *dự báo* trực tiếp.

In [ ]:
inv = R('inventory.csv', parse_dates=['snapshot_date'])
wt  = R('web_traffic.csv', parse_dates=['date'])
rt  = R('returns.csv', parse_dates=['return_date'])
rv  = R('reviews.csv', parse_dates=['review_date'])
sh  = R('shipments.csv', parse_dates=['ship_date'])

print('Phạm vi thời gian các bảng:')
for name, df, col in [('orders',orders,'order_date'), ('inventory',inv,'snapshot_date'),
                      ('web_traffic',wt,'date'), ('returns',rt,'return_date'),
                      ('reviews',rv,'review_date'), ('shipments',sh,'ship_date')]:
    print(f'  {name:12s} {df[col].min().date()} → {df[col].max().date()}   ({len(df):,} dòng)')
print(f'  {"VÙNG DỰ BÁO":12s} 2023-01-01 → 2024-07-01   → KHÔNG bảng nào phủ')

In [ ]:
print('Bẫy & cột chết cần biết:\n')
print('1. web_traffic : 3.652 dòng = ĐÚNG 1 dòng/ngày, nhưng traffic_source có 6 giá trị')
print(f'   → KHÔNG phải phân rã theo nguồn; mỗi ngày mang 1 nhãn xoay vòng. 1 dòng/ngày: {wt.groupby("date").size().max()==1}')
j = sales.merge(wt, left_on='Date', right_on='date')
print(f'   → corr(sessions, Revenue) = {j.sessions.corr(j.Revenue):.3f} — yếu. Bắt đầu 2013-01 (thiếu nửa cuối 2012).')
print(f'   → bounce_rate ≈ {wt.bounce_rate.mean():.4f}: phi thực tế, dữ liệu tổng hợp. Đừng xây feature trên đó.')
print(f'\n2. inventory  : {inv.snapshot_date.nunique()} snapshot = THEO THÁNG (cuối tháng), không phải ngày.')
print(f'   → chỉ phủ {inv.product_id.nunique()}/{prod.product_id.nunique()} sản phẩm. Không join trực tiếp theo ngày được.')
print(f'   → reorder_flag chỉ có 1 giá trị duy nhất ({inv.reorder_flag.nunique()}) — cột chết.')
print(f'\n3. order_items: promo_id_2 rỗng {items.promo_id_2.isna().mean()*100:.2f}% — cột chết.')
print(f'   → discount_amount chỉ chiếm {(items.discount_amount.sum()/(items.quantity*items.unit_price).sum())*100:.2f}% tổng giá trị')
print('     và KHÔNG được trừ khỏi Revenue (đã chứng minh ở phần 1).')
print(f'\n4. returns    : refund_amount KHÔNG được trừ khỏi Revenue.')
print(f'\n5. 2012 chỉ có nửa năm (từ 04/07) — loại khỏi mọi phép tính theo năm.')

---
## 8 — Kết luận & hệ quả cho mô hình

### Lỗi trong `baseline.ipynb`

| Vấn đề | Chi tiết |
|---|---|
| **Không chạy được** | `TEST_FILE` không được định nghĩa; `DATA_DIR='dataset/'` sai với layout phẳng này |
| **MAPE lạc quan giả** | Mục 6 fit seasonal profile trên toàn bộ dữ liệu *rồi* validate trên 2021–2022 → rò rỉ |
| **Tăng trưởng sai** | −3,8%/năm gộp từ 2013–2022, vắt ngang qua đứt gãy 2018. Thực tế 2019+ đi ngang |
| **Bỏ sót ngày-trong-tháng** | Profile theo `(month, day)` có bắt được, nhưng bị pha loãng vì trộn mọi năm |
| **Trộn tháng 8 chẵn/lẻ** | Sai lệch lớn nhất — dự báo thừa ~34% doanh thu T8/2023 |

### Hướng đi có cơ sở từ dữ liệu

1. **Chỉ dùng 2019+ để ước lượng mức**; mùa vụ có thể lấy từ nhiều năm hơn.
2. **Tách riêng profile tháng 8 theo chẵn/lẻ.** 2023 → dùng profile năm **lẻ**
   (chỉ số DT ≈ 0,78 ; COGS/Rev ≈ 1,36). Cửa sổ promo: **30/07/2023 → 02/09/2023**.
   Lưu ý T8/2024 **không** nằm trong vùng dự báo (kết thúc 01/07/2024) nên chỉ cần xử lý 2023.
3. **Dự báo `Revenue` + `COGS/Revenue` ratio**, rồi suy ra COGS — thay vì hai chuỗi độc lập.
4. **Cấu trúc lịch nên là `month × day-of-month`**, bỏ qua thứ-trong-tuần (R² = 0,008).
5. **Bỏ qua các bảng phụ làm feature dự báo** — không bảng nào phủ 2023–2024.
6. Cân nhắc **xu hướng bào mòn biên lợi nhuận** (ratio tăng đều 2020 → 2022) khi ngoại suy.